# ⚡ EV Quantum Forge · Fusion II: Partition-Bagged Recipes (S6E9)

This notebook retrains the strongest *non-blend* public recipes of Playground S6E9 from the data and
fuses them. Its predecessor, the five-recipe fusion (one 5-fold partition), scored **0.94643** public
LB. Every component here is **trained in this notebook** on the competition CSVs and the public
original dataset. No public submission, OOF file or leaderboard CSV is read.

| Recipe | Credit | Their LB | Here |
|---|---|---:|---|
| A | najiama: *XGBoost Triple-TE & Dynamic Pruning* / *Pure LGBM* | 0.94639 / 0.94638 | digits, original means, smooth keys, triple TE → pruned XGB + LGBM |
| C | chinzorigtganbat: *Leak-Safe Features & Logit Blending* | 0.94639 | nested TE, income-neighbourhood channels → LGBM, CatBoost, linear-tree LGBM |
| D | megayak: *Four Feature Views One Ensemble* (view C) | 0.94639 | generator-lift counts + centred window buy-rates → LGBM |
| E | bulsahkecici: *Multi-Res Target Encoding Partition Bag* | 0.94637 | nested exact-value TE, multi-smoothing keys → LGBM |

**What changed since v1**
1. **Partition bagging for every recipe** (idea from recipe E). The full 5-fold CV is repeated with fold
   seeds 42 / 45 / 48 / 51. Each row's OOF is the mean over partitions, and each still comes from
   models that never saw that row. Test predictions average 20 models per recipe instead of 5.
2. **Recipe E** is new. mikhailnaumov's XGB is dropped: v1's blend gave it zero weight, and it was the
   slowest component.
3. The scheduler runs **partition-major**: all recipes finish partition 1 before any starts partition 2.
   If the time budget runs short, every recipe still has complete partitions.
4. The blend is unchanged from v1: Caruana hill-climbing on logits, shipped only if it beats the best
   single recipe on at least 4 of 5 held-out folds.

## 1 · Setup

In [ ]:
from __future__ import annotations

import gc
import glob
import json
import os
import subprocess
import time
import warnings
from dataclasses import dataclass, field
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.ndimage import convolve1d
from scipy.special import expit, logit
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import KBinsDiscretizer, TargetEncoder

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)

T0 = time.time()
SMOKE = os.environ.get("FUSION_SMOKE", "0") == "1"      # tiny local run: code-path check only
TIME_BUDGET_S = float(os.environ.get("FUSION_BUDGET_H", "10.5")) * 3600
BLEND_RESERVE_S = 1200                                   # kept free for blending and export
NFOLD = 5
PARTITION_SEEDS = tuple(int(v) for v in os.environ.get("FUSION_PARTITIONS", "42,45,48,51").split(","))
THREADS = os.cpu_count() or 4
TARGET, ID = "Will_Buy_EV", "id"
CATS = ["Gender", "City_Type", "Current_Car_Type", "Home_Charging_Possible",
        "Subsidy_Available", "Range_Anxiety_Level"]
NUMS = ["Age", "Annual_Income_USD", "Daily_Commute_km", "Number_of_Cars_Owned",
        "Charging_Stations_Near_Home", "Charging_Stations_Near_Work", "Environmental_Concern_Level"]
OUT = Path("fusion_outputs")
OUT.mkdir(exist_ok=True)


def elapsed() -> str:
    return f"{(time.time() - T0) / 60:6.1f} min"


def remaining() -> float:
    return TIME_BUDGET_S - (time.time() - T0)


def find(name: str) -> str | None:
    for root in (os.environ.get("FUSION_DATA", ""), "/kaggle/input", "data", "../data"):
        if root and Path(root).exists():
            hits = sorted(glob.glob(f"{root}/**/{name}", recursive=True))
            if hits:
                return hits[0]
    return None


def gpu_usable() -> bool:
    if os.environ.get("FUSION_FORCE_CPU") == "1":
        return False
    try:
        subprocess.check_output("nvidia-smi", stderr=subprocess.DEVNULL, shell=True)
        probe = np.random.rand(600, 4)
        xgb.XGBClassifier(n_estimators=5, max_depth=3, tree_method="hist", device="cuda",
                          verbosity=0).fit(probe, (probe[:, 0] > 0.5).astype(int))
        return True
    except Exception as exc:  # noqa: BLE001 - any failure means CPU fallback
        print(f"GPU not usable for XGBoost ({type(exc).__name__}); XGBoost runs on CPU")
        return False


GPU = gpu_usable()
XGB_DEVICE = "cuda" if GPU else "cpu"

# Round caps. Smoke mode only checks the code paths.
R = dict(xgb=200, lgb=200, es=20, cat=40, fixed_scale=0.03) if SMOKE else \
    dict(xgb=100_000, lgb=20_000, es=500, cat=None, fixed_scale=1.0)

## 2 · Data and the fold partitions

In [ ]:
train_raw = pd.read_csv(find("train.csv"))
test_raw = pd.read_csv(find("test.csv"))
sample = pd.read_csv(find("sample_submission.csv"))
orig_path = find("EV_Adoption_and_Range_Anxiety_Dataset.csv")
orig_raw = pd.read_csv(orig_path) if orig_path else None

if SMOKE:
    train_raw = train_raw.sample(40_000, random_state=0).reset_index(drop=True)
    test_raw = test_raw.head(15_000).reset_index(drop=True)
    sample = sample[sample[ID].isin(test_raw[ID])].reset_index(drop=True)
    if orig_raw is None:  # local stand-in so every code path runs; never used on Kaggle
        orig_raw = pd.read_csv(find("train.csv"), nrows=8_000).rename(columns={ID: "Buyer_ID"})
        print("SMOKE: original dataset missing, using a train slice as a stand-in")

y = (train_raw[TARGET] == "Yes").astype(np.int8).to_numpy()
test_ids = test_raw[ID].to_numpy()
if SMOKE:
    PARTITION_SEEDS = PARTITION_SEEDS[:2]
FOLDS_BY_SEED = {seed: list(StratifiedKFold(NFOLD, shuffle=True, random_state=seed).split(np.zeros(len(y)), y))
                 for seed in PARTITION_SEEDS}
GATE_FOLDS = FOLDS_BY_SEED[PARTITION_SEEDS[0]]      # partition 1 folds drive the nested blend gate
fold_id = np.empty(len(y), dtype=np.int8)
for k, (_, va) in enumerate(GATE_FOLDS):
    fold_id[va] = k

print(f"train {train_raw.shape} | test {test_raw.shape} | original "
      f"{None if orig_raw is None else orig_raw.shape} | positives {y.mean():.4%}")
print(f"GPU for XGBoost: {GPU} | threads {THREADS} | smoke {SMOKE} | budget {TIME_BUDGET_S / 3600:.1f} h"
      f" | partitions {PARTITION_SEEDS}")

## 3 · Partition scheduler

Recipes register a target-free `prepare()` and a `fold_fn`. `prepare()` runs once and is cached. The
scheduler loops **partition → recipe → fold**. A recipe starts a new partition only if the time left
covers 1.25× its previous partition time, with a reserve kept for blending. A partition that cannot
finish is dropped as a whole.

In [ ]:
RESULTS: dict[str, dict] = {}
GROUPS: list[dict] = []


def register(title: str, prepare, fold_fn, first_partition_min: float = 30):
    """prepare() -> ctx ; fold_fn(ctx, k, tr, va) -> {name: (valid_pred, test_pred, info)}"""
    GROUPS.append(dict(title=title, prepare=prepare, fold_fn=fold_fn, ctx=None, last_s=None,
                       min_s=first_partition_min * 60, active=True))


def run_partition(group: dict, seed: int):
    need = 1.25 * group["last_s"] if group["last_s"] else group["min_s"]
    if remaining() < need + BLEND_RESERVE_S:
        print(f"[stop] {group['title']}: {remaining() / 60:.0f} min left, a partition needs ~{need / 60:.0f}")
        group["active"] = False
        return
    print(f"\n--- {group['title']} | partition seed {seed}   (start {elapsed()})")
    if group["ctx"] is None:
        t_prep = time.time()
        group["ctx"] = group["prepare"]()
        print(f"  features ready in {time.time() - t_prep:.0f}s")
    t_part = time.time()
    oof, tst, fold_auc = {}, {}, {}
    for k, (tr, va) in enumerate(FOLDS_BY_SEED[seed]):
        if remaining() < 0.5 * BLEND_RESERVE_S:
            print(f"  [abort] {group['title']}: out of time inside partition {seed}; partition discarded")
            group["active"] = False
            return
        t1 = time.time()
        for name, (pv, pt, info) in group["fold_fn"](group["ctx"], k, tr, va).items():
            oof.setdefault(name, np.zeros(len(y)))[va] = pv
            tst.setdefault(name, np.zeros(len(test_ids)))
            tst[name] += pt / NFOLD
            fold_auc.setdefault(name, []).append(roc_auc_score(y[va], pv))
            print(f"  fold {k} | {name:<22} auc {fold_auc[name][-1]:.6f} | {info} | {time.time() - t1:.0f}s")
        gc.collect()
    group["last_s"] = time.time() - t_part
    for name in oof:
        r = RESULTS.setdefault(name, dict(oofs=[], tests=[], seeds=[], partition_auc=[], fold_auc=None, seconds=0.0))
        r["oofs"].append(oof[name])
        r["tests"].append(tst[name])
        r["seeds"].append(seed)
        r["partition_auc"].append(float(roc_auc_score(y, oof[name])))
        r["fold_auc"] = r["fold_auc"] or fold_auc[name]
        r["seconds"] += group["last_s"]
        bag = roc_auc_score(y, np.mean(r["oofs"], axis=0))
        print(f"  >>> {name:<22} partition AUC {r['partition_auc'][-1]:.6f} | bag of {len(r['oofs'])}: {bag:.6f}")


def codes(values) -> np.ndarray:
    """Category identity as int32. TargetEncoder treats every input as categorical, so this matches
    string keys exactly and uses far less memory."""
    return pd.factorize(pd.Series(values), use_na_sentinel=False)[0].astype(np.int32)


def te_frame(K_tr, y_tr, K_others, smooths, names, style):
    """sklearn TargetEncoder (cross-fitted on the fit rows). style='per_col' interleaves the smoothing
    levels per column (najiama order); 'per_smooth' groups by smoothing (mikhailnaumov order)."""
    blocks_tr, blocks_ot = {}, [dict() for _ in K_others]
    for sm in smooths:
        tag = sm if isinstance(sm, str) else str(int(sm)) if float(sm).is_integer() else str(sm)
        enc = TargetEncoder(shuffle=True, cv=5, smooth=sm, random_state=42)
        a = enc.fit_transform(K_tr, y_tr).astype(np.float32)
        others = [enc.transform(K).astype(np.float32) for K in K_others]
        for i, col in enumerate(names):
            blocks_tr[(col, tag)] = a[:, i]
            for j, o in enumerate(others):
                blocks_ot[j][(col, tag)] = o[:, i]
    tags = [sm if isinstance(sm, str) else str(int(sm)) if float(sm).is_integer() else str(sm)
            for sm in smooths]
    order = ([(c, t) for c in names for t in tags] if style == "per_col"
             else [(c, t) for t in tags for c in names])
    fmt = "{}_TE_{}" if style == "per_col" else "{}_TE_s{}"
    cols = [fmt.format(c, t) for c, t in order]
    to_df = lambda blk, n: pd.DataFrame({cn: blk[key] for cn, key in zip(cols, order)}, index=range(n))
    return to_df(blocks_tr, len(K_tr)), [to_df(b, len(K)) for b, K in zip(blocks_ot, K_others)]

## 4 · Recipe A: najiama feature forge → XGB (pruned) + LGBM

The source notebooks train XGB and LGBM separately on identical features. Here they share each
fold's triple-TE block. `Number_of_Cars_Owned` is dropped, as in the source. The XGB prune list is
copied verbatim.

In [ ]:
NAJIAMA_PRUNE = [
    'Age', 'Daily_Commute_km_digit-3_cat_TE_100', 'Charging_Stations_Near_Home_digit0',
    'Daily_Commute_km_digit1_cat_TE_100', 'Annual_Income_USD_digit1', 'Charging_Stations_Near_Work_digit1_cat_TE_auto',
    'Daily_Commute_km_digit1_cat_TE_auto', 'Daily_Commute_km_digit0_cat_TE_10', 'Gender_org_mean',
    'Daily_Commute_km_digit-4_cat_TE_100', 'Daily_Commute_km_digit0_cat_TE_auto', 'Daily_Commute_km_digit-4_cat_TE_auto',
    'Daily_Commute_km_cat_fe', 'Annual_Income_USD_digit1_cat_fe', 'Charging_Stations_Near_Work_digit0_cat_TE_100',
    'Annual_Income_USD_digit2_cat_TE_100', 'Annual_Income_USD_digit0', 'Charging_Stations_Near_Home_digit-3_cat_TE_auto',
    'Gender_TE_100', 'Current_Car_Type_TE_10', 'Age_digit1_cat_fe', 'Charging_Stations_Near_Work_digit1_cat_TE_100',
    'Annual_Income_USD_digit0_cat_TE_10', 'Daily_Commute_km_digit-3_cat_TE_10', 'Charging_Stations_Near_Work_digit0_cat_TE_10',
    'Gender_TE_auto', 'Daily_Commute_km_digit0_cat_TE_100', 'Charging_Stations_Near_Home_digit-4_cat_TE_auto',
    'Daily_Commute_km_digit-1_cat_TE_10', 'Charging_Stations_Near_Work_digit0_cat_TE_auto', 'Daily_Commute_km_digit-1',
    'Charging_Stations_Near_Work_digit0', 'Annual_Income_USD_digit0_cat_TE_auto', 'Daily_Commute_km_digit-4_cat_TE_10',
    'Charging_Stations_Near_Work_digit-3_cat_TE_auto', 'Charging_Stations_Near_Work_org_mean', 'Age_digit0_cat_TE_auto',
    'Age_digit0_cat_fe', 'Annual_Income_USD_digit0_cat_fe', 'Charging_Stations_Near_Home_digit-4_cat_TE_100',
    'Daily_Commute_km_digit-3_cat_TE_auto', 'Age_digit0', 'Annual_Income_USD_digit2_cat_TE_auto',
    'Annual_Income_USD_digit0_cat_TE_100', 'Daily_Commute_km_org_mean', 'Age_digit0_cat_TE_10',
    'Current_Car_Type_TE_auto', 'Current_Car_Type_TE_100', 'Age_digit1_cat_TE_auto', 'Charging_Stations_Near_Home_digit-4',
    'Annual_Income_USD_digit2_cat_TE_10', 'Daily_Commute_km_digit-1_cat_TE_100', 'Age_cat_fe',
    'Daily_Commute_km_digit0_cat_fe', 'Daily_Commute_km_digit-1_cat_fe', 'Daily_Commute_km_digit-1_cat_TE_auto',
    'Daily_Commute_km_digit-2_cat_TE_10', 'Age_digit1', 'Age_digit1_cat_TE_100', 'Gender_TE_10', 'Age_digit1_cat_TE_10',
    'Age_digit0_cat_TE_100', 'Daily_Commute_km_digit-3', 'Charging_Stations_Near_Home_digit-3_cat_TE_100', 'Gender_fe',
    'Charging_Stations_Near_Home_digit-3_cat_TE_10', 'Daily_Commute_km_digit-4_cat_fe', 'Daily_Commute_km_digit-4',
    'Daily_Commute_km_digit-3_cat_fe', 'Charging_Stations_Near_Work_digit-3_cat_TE_10', 'Daily_Commute_km_digit-2_cat_TE_100',
    'Charging_Stations_Near_Home_digit-2_cat_TE_auto', 'Charging_Stations_Near_Home_digit-1_cat_TE_auto',
    'Charging_Stations_Near_Work_digit1_cat_fe', 'Charging_Stations_Near_Work_digit-2_cat_TE_100',
    'Charging_Stations_Near_Home_digit-2_cat_TE_100', 'Charging_Stations_Near_Work_digit-3_cat_TE_100', 'is_dead_zone',
    'is_millionaire_cliff', 'is_30k_spike', 'Charging_Stations_Near_Home_digit-1_cat_TE_100',
    'Charging_Stations_Near_Home_digit-1_cat_TE_10', 'Charging_Stations_Near_Home_digit-2_cat_TE_10',
    'Charging_Stations_Near_Work_digit-1_cat_TE_10', 'Charging_Stations_Near_Work_digit-2_cat_TE_auto',
    'Charging_Stations_Near_Work_digit-1_cat_TE_auto', 'Charging_Stations_Near_Work_digit-1_cat_TE_100',
    'Charging_Stations_Near_Work_digit-2_cat_TE_10',
]


def prepare_najiama():
    tr = train_raw.drop(columns=[TARGET]).copy()
    comb = pd.concat([tr, test_raw], ignore_index=True).drop(columns=["Number_of_Cars_Owned"])
    n = len(tr)
    cat_cols = [c for c in comb.columns if not pd.api.types.is_numeric_dtype(comb[c])]
    num_cols = [c for c in comb.columns if c not in cat_cols + [ID]]
    new = {}
    digit_cols = []
    for c in num_cols:
        v = comb[c].fillna(0)
        for k in range(-4, 4):
            name = f"{c}_digit{k}"
            new[name] = (v // (10 ** k) % 10).astype("int8")
            digit_cols.append(name)
    comb = pd.concat([comb, pd.DataFrame(new)], axis=1)
    num_cols = num_cols + digit_cols
    if orig_raw is not None:
        og = orig_raw.copy()
        og[TARGET] = og[TARGET].map({"Yes": 1, "No": 0})
        gm = og[TARGET].mean()
        for col in cat_cols + num_cols:
            if col in og.columns:
                comb[f"{col}_org_mean"] = comb[col].map(og.groupby(col, observed=False)[TARGET].mean()).fillna(gm).astype(float)
    keys = {c: codes(comb[c]) for c in cat_cols}
    for col in num_cols:
        keys[f"{col}_cat"] = codes(comb[col])
    all_cats = list(keys)
    for col in all_cats:
        s = pd.Series(keys[col])
        comb[f"{col}_fe"] = s.map(s.value_counts(normalize=True)).astype(float).to_numpy()
    inc = comb["Annual_Income_USD"]
    comb["is_30k_spike"] = (inc == 30000.0).astype("int8")
    comb["is_millionaire_cliff"] = (inc >= 170537.0).astype("int8")
    comb["is_dead_zone"] = ((inc >= 38000.0) & (inc <= 42000.0)).astype("int8")
    comb["is_env_hater"] = (comb["Environmental_Concern_Level"] == 1).astype("int8")
    keys["income_exact_int"] = codes(np.floor(inc))
    keys["income100_floor"] = codes(np.floor(inc / 100.0))
    keys["income1000_floor"] = codes(np.floor(inc / 1000.0))
    keys["commute_integer"] = codes(np.floor(comb["Daily_Commute_km"]))
    num = comb.drop(columns=cat_cols + [ID])
    K = pd.DataFrame(keys)
    X, Xt, Ktr, Kte = num.iloc[:n].reset_index(drop=True), num.iloc[n:].reset_index(drop=True), \
        K.iloc[:n].reset_index(drop=True), K.iloc[n:].reset_index(drop=True)
    # Drop perfectly correlated numeric columns (train) and constants (train or test), as in the source.
    corr = X.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    drop = {c for c in upper.columns if any(upper[c] == 1.0)}
    drop |= {c for c in X.columns if X[c].nunique() == 1 or Xt[c].nunique() == 1}
    kdrop = {c for c in K.columns if Ktr[c].nunique() == 1 or Kte[c].nunique() == 1}
    X, Xt = X.drop(columns=sorted(drop)), Xt.drop(columns=sorted(drop))
    Ktr, Kte = Ktr.drop(columns=sorted(kdrop)), Kte.drop(columns=sorted(kdrop))
    print(f"  najiama: {X.shape[1]} numeric features, {Ktr.shape[1]} TE keys, dropped {len(drop) + len(kdrop)}")
    return dict(X=X, Xt=Xt, K=Ktr, Kt=Kte)


def fold_najiama(ctx, k, tr, va):
    X, Xt, K, Kt = ctx["X"], ctx["Xt"], ctx["K"], ctx["Kt"]
    names = list(K.columns)
    te_a, (te_b, te_c) = te_frame(K.iloc[tr].to_numpy(), y[tr], [K.iloc[va].to_numpy(), Kt.to_numpy()],
                                  ("auto", 10.0, 100.0), names, "per_col")
    A = pd.concat([X.iloc[tr].reset_index(drop=True), te_a], axis=1)
    B = pd.concat([X.iloc[va].reset_index(drop=True), te_b], axis=1)
    C = pd.concat([Xt, te_c], axis=1)
    out = {}
    # XGB (najiama triple-TE & dynamic pruning), pruned columns removed
    keep = [c for c in A.columns if c not in set(NAJIAMA_PRUNE)]
    m = xgb.XGBClassifier(
        n_estimators=R["xgb"], learning_rate=0.01, max_depth=4, min_child_weight=4.532387806880492,
        subsample=0.7400402414525654, colsample_bytree=0.5695776529558766, alpha=0.7523885021652775,
        reg_lambda=0.6189949691705282, max_bin=1024, random_state=42, eval_metric="auc",
        early_stopping_rounds=R["es"], tree_method="hist", device=XGB_DEVICE, n_jobs=THREADS,
        objective="binary:logistic", max_leaves=16, grow_policy="lossguide", gamma=3.673225596759869)
    m.fit(A[keep], y[tr], eval_set=[(B[keep], y[va])], verbose=False)
    out["A1_najiama_xgb"] = (m.predict_proba(B[keep])[:, 1], m.predict_proba(C[keep])[:, 1],
                             f"trees {m.best_iteration} cols {len(keep)}")
    del m
    # LGBM (najiama pure LGBM), all columns
    m = lgb.LGBMClassifier(
        n_estimators=R["lgb"], learning_rate=0.02, max_depth=5, num_leaves=32, min_child_samples=10,
        subsample=0.812763, subsample_freq=0, colsample_bytree=0.30293, reg_alpha=0.07094, reg_lambda=2.03303,
        max_bin=1024, random_state=42, feature_pre_filter=False, metric="auc", n_jobs=THREADS, verbose=-1)
    m.fit(A, y[tr], eval_set=[(B, y[va])], callbacks=[lgb.early_stopping(R["es"], verbose=False)])
    out["A2_najiama_lgbm"] = (m.predict_proba(B)[:, 1], m.predict_proba(C)[:, 1],
                              f"trees {m.best_iteration_} cols {A.shape[1]}")
    return out


register("Recipe A · najiama forge → XGB + LGBM", prepare_najiama, fold_najiama, first_partition_min=25)

## 5 · Recipe C: chinzorigtganbat leak-safe pipelines

The feature and model pipelines are ported from the source. All target statistics are nested with an
inner KFold. The three members are LGBM `smooth1_bagging`, CatBoost with income-neighbourhood
channels, and linear-tree LGBM. Each uses the source's frozen round counts, with no early stopping
on the outer fold.

In [ ]:
RAW = NUMS + CATS
CATEGORIES = {
    'Gender': ['Male', 'Female', 'Other'], 'City_Type': ['Urban', 'Suburban', 'Rural'],
    'Current_Car_Type': ['Sedan', 'SUV', 'Hatchback', 'Truck'],
    'Home_Charging_Possible': ['No', 'Yes'], 'Subsidy_Available': ['No', 'Yes'],
    'Range_Anxiety_Level': ['Low', 'Medium', 'High'],
}


@dataclass
class FeatureConfig:
    smooth_keys: bool = False
    digits: bool = True
    frequency: bool = True
    target_encoding: bool = True
    te_smoothing: tuple = ('auto', 10.0)
    te_columns: tuple = ()
    inner_folds: int = 5
    inner_seed: int = 17
    income_bins: tuple = ()
    neighborhood_smoothing: float = 10.0
    neighborhood_channels: tuple = ('central', 'neighbor', 'slope', 'curvature', 'count')
    income_kernel_sigma: float = 0.0
    scale_numeric: bool = False
    compact: bool = False


class FeaturePipeline:
    """Target-free state is fitted once; supervised encodings are cross-fitted on the fit rows."""

    def __init__(self, config: FeatureConfig):
        self.config = config

    def _base(self, X):
        c = self.config
        f = X[RAW].copy().reset_index(drop=True)
        for col in NUMS:
            f[col] = pd.to_numeric(f[col], errors='coerce').fillna(self.medians_[col])
        for col, levels in CATEGORIES.items():
            f[col] = f[col].map({v: i for i, v in enumerate(levels)}).fillna(-1).astype('int16')
        income = np.rint(f.Annual_Income_USD).astype('int64')
        commute = np.rint(10 * f.Daily_Commute_km).astype('int64')
        if c.digits:
            for power in range(5):
                f[f'income_digit_{power}'] = (income // 10 ** power) % 10
            f['commute_tenths'] = commute % 10
            f['commute_units'] = (commute // 10) % 10
        if c.compact:
            f['homecharge'] = f.Charging_Stations_Near_Home + 5 * f.Home_Charging_Possible
        if c.smooth_keys:
            f['smooth_key_income100'] = income // 100
            f['smooth_key_income1000'] = income // 1000
            f['smooth_key_commute_integer'] = commute // 10
        return f

    def _fit_supervised(self, base, yv):
        c = self.config
        state = {'prior': float(np.mean(yv)), 'encoders': [], 'bins': []}
        if c.target_encoding:
            keys = base[self.te_columns_].astype(str)
            for smoothing in c.te_smoothing:
                enc = TargetEncoder(smooth=smoothing, target_type='binary')
                enc.fit(keys, yv)
                state['encoders'].append(enc)
        income = base.Annual_Income_USD.to_numpy(float)
        for bins in c.income_bins:
            edges = self.income_edges_[bins]
            cds = np.searchsorted(edges[1:-1], income)
            sums = np.bincount(cds, weights=yv, minlength=bins)
            counts = np.bincount(cds, minlength=bins).astype(float)
            smooth = c.neighborhood_smoothing
            central = (sums + smooth * state['prior']) / (counts + smooth)
            weights = np.exp(-.5 * (np.arange(-2, 3) / .8) ** 2)
            neighbor = (convolve1d(sums, weights, mode='constant', cval=0) + smooth * state['prior']) / \
                       (convolve1d(counts, weights, mode='constant', cval=0) + smooth)
            left = np.r_[state['prior'], central[:-1]]
            right = np.r_[central[1:], state['prior']]
            values = dict(central=central, neighbor=neighbor, left=left, right=right, slope=right - left,
                          curvature=central - .5 * (left + right), count=np.log1p(counts))
            state['bins'].append((edges, values))
        if c.income_kernel_sigma > 0:
            vals = np.rint(income).astype('int64')
            lo, hi = int(vals.min()), int(vals.max())
            counts = np.bincount(vals - lo, minlength=hi - lo + 1).astype(float)
            sums = np.bincount(vals - lo, weights=yv, minlength=hi - lo + 1)
            radius = max(1, int(np.ceil(8 * c.income_kernel_sigma)))
            kernel = np.exp(-.5 * (np.arange(-radius, radius + 1) / c.income_kernel_sigma) ** 2)
            num = convolve1d(sums, kernel, mode='constant', cval=0)
            den = convolve1d(counts, kernel, mode='constant', cval=0)
            state['kernel'] = (lo, (num + c.neighborhood_smoothing * state['prior']) / (den + c.neighborhood_smoothing))
        return state

    def _supervised(self, base, state):
        blocks, names = [], []
        if state['encoders']:
            keys = base[self.te_columns_].astype(str)
            for i, enc in enumerate(state['encoders']):
                blocks.append(enc.transform(keys))
                names += [f'te{i}__{col}' for col in self.te_columns_]
        income = base.Annual_Income_USD.to_numpy(float)
        for q, (edges, values) in zip(self.config.income_bins, state['bins']):
            cds = np.searchsorted(edges[1:-1], income)
            for channel in self.config.neighborhood_channels:
                blocks.append(values[channel][cds, None])
                names.append(f'income_bin{q}__{channel}')
        if 'kernel' in state:
            lo, lookup = state['kernel']
            cds = np.rint(income).astype('int64') - lo
            valid = (cds >= 0) & (cds < len(lookup))
            result = np.full(len(base), state['prior'])
            result[valid] = lookup[cds[valid]]
            blocks.append(result[:, None])
            names.append('income_kernel')
        arr = np.column_stack(blocks).astype('float32') if blocks else np.empty((len(base), 0), dtype='float32')
        return pd.DataFrame(arr, columns=names)

    def _unsupervised(self, base):
        c = self.config
        columns = (['Annual_Income_USD', 'Daily_Commute_km', 'Charging_Stations_Near_Work',
                    'Environmental_Concern_Level', 'Home_Charging_Possible', 'Subsidy_Available',
                    'Range_Anxiety_Level', 'homecharge'] if c.compact else list(base.columns))
        out = base[columns].copy().astype('float32')
        if c.scale_numeric:
            out = (out - self.scale_mean_[columns]) / self.scale_std_[columns]
        if c.frequency:
            for col, mapping in self.frequencies_.items():
                out[f'freq__{col}'] = base[col].map(mapping).fillna(0).astype('float32')
        return out

    def fit_transform(self, X, yv):
        c = self.config
        self.medians_ = X[NUMS].median().fillna(0)
        base = self._base(X)
        income = base.Annual_Income_USD.to_numpy(float)
        lo, hi = float(income.min()), float(income.max())
        self.income_edges_ = {q: np.linspace(lo, max(lo + 1, hi), q + 1) for q in c.income_bins}
        self.te_columns_ = list(c.te_columns) if c.te_columns else [
            col for col in base if col in RAW or 'digit' in col or 'commute_' in col
            or 'income_mod_' in col or 'smooth_key_' in col]
        self.scale_mean_ = base.mean()
        self.scale_std_ = base.std(ddof=0).replace(0, 1).fillna(1)
        self.frequencies_ = {col: base[col].value_counts(normalize=True) for col in self.te_columns_} if c.frequency else {}
        self.supervised_state_ = self._fit_supervised(base, np.asarray(yv))
        template = self._supervised(base.iloc[:1], self.supervised_state_)
        encoded = np.empty((len(base), template.shape[1]), dtype='float32')
        for fit, hold in KFold(c.inner_folds, shuffle=True, random_state=c.inner_seed).split(base):
            state = self._fit_supervised(base.iloc[fit], np.asarray(yv)[fit])
            encoded[hold] = self._supervised(base.iloc[hold], state).to_numpy()
        result = pd.concat([self._unsupervised(base), pd.DataFrame(encoded, columns=template.columns)], axis=1)
        self.columns_ = [col for col in result if col in template.columns or result[col].nunique(dropna=False) > 1]
        return result[self.columns_]

    def transform(self, X):
        base = self._base(X)
        return pd.concat([self._unsupervised(base), self._supervised(base, self.supervised_state_)], axis=1)[self.columns_]


LGB_BASE = dict(objective='binary', metric='auc', learning_rate=.02, max_depth=5, num_leaves=31,
                min_data_in_leaf=10, feature_fraction=.30, lambda_l1=.071, lambda_l2=2.0, max_bin=1024,
                feature_pre_filter=False, verbosity=-1, seed=42, num_threads=THREADS)
CHINZO = {
    "C1_chinzo_lgbm": dict(
        features=FeatureConfig(income_bins=(8192, 16384), neighborhood_smoothing=1.0, smooth_keys=True,
                               te_smoothing=('auto', 10.0, 100.0)),
        learner="lightgbm", rounds=1377, params=dict(LGB_BASE, bagging_fraction=0.85, bagging_freq=1)),
    "C2_chinzo_catboost": dict(
        features=FeatureConfig(income_bins=(8192, 16384), smooth_keys=True, te_smoothing=('auto', 10.0, 100.0)),
        learner="catboost", rounds=944,
        params=dict(loss_function='Logloss', depth=5, learning_rate=0.04, l2_leaf_reg=5.0, rsm=0.5)),
    "C3_chinzo_linear": dict(
        features=FeatureConfig(digits=False, te_smoothing=(0.5,), te_columns=(
            'Age', 'Gender', 'Number_of_Cars_Owned', 'City_Type', 'Current_Car_Type', 'Range_Anxiety_Level'),
            neighborhood_smoothing=0.5, income_kernel_sigma=5.0, scale_numeric=True, compact=True),
        learner="lightgbm", rounds=967,
        params=dict(LGB_BASE, max_depth=-1, bagging_fraction=0.9, bagging_freq=1, lambda_l1=0.05, lambda_l2=10,
                    max_bin=1023, linear_tree=True, linear_lambda=0.5)),
}


def chinzo_member(name):
    cfg = CHINZO[name]

    def prepare():
        return dict(X=train_raw[RAW], Xt=test_raw[RAW])

    def fold(ctx, k, tr, va):
        prep = FeaturePipeline(cfg["features"])
        a = prep.fit_transform(ctx["X"].iloc[tr], y[tr])
        b, t = prep.transform(ctx["X"].iloc[va]), prep.transform(ctx["Xt"])
        rounds = max(20, int(cfg["rounds"] * R["fixed_scale"]))
        if cfg["learner"] == "lightgbm":
            m = lgb.train(cfg["params"], lgb.Dataset(a, label=y[tr]), num_boost_round=rounds)
            pv, pt = m.predict(b), m.predict(t)
        else:
            from catboost import CatBoostClassifier
            m = CatBoostClassifier(**cfg["params"], iterations=rounds, thread_count=THREADS, random_seed=42,
                                   verbose=False, allow_writing_files=False)
            m.fit(a, y[tr])
            pv, pt = m.predict_proba(b)[:, 1], m.predict_proba(t)[:, 1]
        return {name: (pv, pt, f"rounds {rounds} cols {a.shape[1]}")}

    return prepare, fold


for member in CHINZO:
    register(f"Recipe C · {member}", *chinzo_member(member), first_partition_min=20)

## 6 · Recipe D: megayak view C (generator lift + window rates)

`lift` measures how much the synthetic generator over-produced an income or commute value compared
with the original table. Window rates are smoothed buy-rates over centred value windows. Both are
nested inside each fold. TE smoothing is 2 / 30 / 300, as in the source's view C.

In [ ]:
WIN_INC, WIN_KM, WIN_GRP, WIN_PRIOR = (2, 5, 10, 25, 50, 200), (1, 3, 10), (5, 25), 10.0
WIN_COLS = ([f"win_inc_{w}" for w in WIN_INC] + [f"win_km_{w}" for w in WIN_KM]
            + [f"win_inc_{w}_citycar" for w in WIN_GRP])


def window_rates(v_fit, y_fit, v_query, widths, prior, gm):
    lo, hi = int(min(v_fit.min(), v_query.min())), int(max(v_fit.max(), v_query.max()))
    n = hi - lo + 1
    cnt = np.bincount(v_fit - lo, minlength=n).astype(np.float64)
    s = np.bincount(v_fit - lo, weights=y_fit, minlength=n)
    ccnt = np.concatenate([[0.0], np.cumsum(cnt)])
    cs = np.concatenate([[0.0], np.cumsum(s)])
    q = v_query - lo
    out = []
    for w in widths:
        a = np.clip(q - w, 0, n)
        b = np.clip(q + w + 1, 0, n)
        out.append((((cs[b] - cs[a]) + prior * gm) / ((ccnt[b] - ccnt[a]) + prior)).astype("float32"))
    return np.stack(out, 1)


def window_block(inc_a, km_a, y_a, inc_q, km_q, g_a, g_q):
    gm = float(y_a.mean())
    parts = [window_rates(inc_a, y_a, inc_q, WIN_INC, WIN_PRIOR, gm),
             window_rates(km_a, y_a, km_q, WIN_KM, WIN_PRIOR, gm)]
    out = np.full((len(inc_q), len(WIN_GRP)), gm, np.float32)
    for g in np.unique(g_q):
        ma, mq = g_a == g, g_q == g
        if ma.sum():
            out[mq] = window_rates(inc_a[ma], y_a[ma], inc_q[mq], WIN_GRP, WIN_PRIOR, gm)
    parts.append(out)
    return np.concatenate(parts, 1)


def prepare_megayak():
    trn = train_raw.drop(columns=[ID, TARGET])
    tst = test_raw.drop(columns=[ID])
    n = len(trn)
    df = pd.concat([trn, tst], ignore_index=True)
    inc = df.Annual_Income_USD.to_numpy(np.int64)
    km = np.round(df.Daily_Commute_km.to_numpy(float) * 10).astype(np.int64)
    df["is_30k_spike"] = (inc == 30000).astype("int8")
    df["is_millionaire_cliff"] = (inc >= 170537).astype("int8")
    df["is_dead_zone"] = ((inc >= 38000) & (inc <= 42000)).astype("int8")
    df["is_env_hater"] = (df.Environmental_Concern_Level == 1).astype("int8")
    if orig_raw is not None:
        og = orig_raw.dropna(subset=["Annual_Income_USD", "Daily_Commute_km", "Environmental_Concern_Level"]).copy()
        og[TARGET] = (og[TARGET] == "Yes").astype(int)
        gm = og[TARGET].mean()
        for c in CATS + NUMS:
            df[f"{c}_org_mean"] = df[c].map(og.groupby(c)[TARGET].mean()).fillna(gm).astype("float32")
        oi = np.round(orig_raw.Annual_Income_USD.dropna().to_numpy(float)).astype(np.int64)
        ok = np.round(orig_raw.Daily_Commute_km.dropna().to_numpy(float) * 10).astype(np.int64)
        for name, comp, src, divs in (("inc", inc, oi, (1, 100, 1000)), ("km", km, ok, (1, 10))):
            for d in divs:
                c_s, o_s = pd.Series(comp // d), pd.Series(src // d)
                c_cnt = c_s.map(c_s.value_counts()).to_numpy(float)
                o_cnt = c_s.map(o_s.value_counts()).fillna(0).to_numpy(float)
                sfx = "" if d == 1 else f"_q{d}"
                df[f"lift_{name}{sfx}"] = (np.log((c_cnt + 1) / len(df)) - np.log((o_cnt + 1) / len(src))).astype("float32")
                df[f"ocnt_{name}{sfx}"] = o_cnt.astype("float32")
                if d == 1:
                    df[f"novel_{name}"] = (o_cnt == 0).astype("int8")
    te = {"k_inc_exact": inc, "k_inc100": inc // 100, "k_inc1000": inc // 1000, "k_km_int": km // 10}
    for c in CATS:
        te[f"k_{c}"] = df[c].astype(str).to_numpy()
    for c in ("Age", "Number_of_Cars_Owned", "Charging_Stations_Near_Home",
              "Charging_Stations_Near_Work", "Environmental_Concern_Level"):
        te[f"k_{c}"] = df[c].astype(str).to_numpy()
    K = pd.DataFrame({c: codes(v) for c, v in te.items()})
    for c in K.columns:
        df[f"{c}_fe"] = K[c].map(K[c].value_counts(normalize=True)).astype("float32").to_numpy()
    for c in CATS:
        df[c] = df[c].astype("category")
    g_all = pd.factorize(df.City_Type.astype(str) + "|" + df.Current_Car_Type.astype(str))[0]
    return dict(X=df.iloc[:n].reset_index(drop=True), Xt=df.iloc[n:].reset_index(drop=True),
                K=K.iloc[:n].reset_index(drop=True), Kt=K.iloc[n:].reset_index(drop=True),
                inc_tr=inc[:n], inc_te=inc[n:], km_tr=km[:n], km_te=km[n:], g_tr=g_all[:n], g_te=g_all[n:])


def fold_megayak(ctx, k, a, b):
    X, Xt, K, Kt = ctx["X"], ctx["Xt"], ctx["K"], ctx["Kt"]
    A, B, C = X.iloc[a].reset_index(drop=True), X.iloc[b].reset_index(drop=True), Xt.copy()
    for sm in (2.0, 30.0, 300.0):
        tag = str(int(sm))
        enc = TargetEncoder(shuffle=True, cv=5, smooth=sm, random_state=42)
        ea = enc.fit_transform(K.iloc[a].to_numpy(), y[a])
        eb, ec = enc.transform(K.iloc[b].to_numpy()), enc.transform(Kt.to_numpy())
        for i, c in enumerate(K.columns):
            A[f"{c}_te{tag}"] = ea[:, i].astype("float32")
            B[f"{c}_te{tag}"] = eb[:, i].astype("float32")
            C[f"{c}_te{tag}"] = ec[:, i].astype("float32")
    inc_tr, km_tr, g_tr = ctx["inc_tr"], ctx["km_tr"], ctx["g_tr"]
    wa = np.zeros((len(a), len(WIN_COLS)), np.float32)
    for ia, ib in StratifiedKFold(5, shuffle=True, random_state=42).split(a, y[a]):
        wa[ib] = window_block(inc_tr[a[ia]], km_tr[a[ia]], y[a[ia]].astype(float),
                              inc_tr[a[ib]], km_tr[a[ib]], g_tr[a[ia]], g_tr[a[ib]])
    wb = window_block(inc_tr[a], km_tr[a], y[a].astype(float), inc_tr[b], km_tr[b], g_tr[a], g_tr[b])
    wc = window_block(inc_tr[a], km_tr[a], y[a].astype(float), ctx["inc_te"], ctx["km_te"], g_tr[a], ctx["g_te"])
    for i, c in enumerate(WIN_COLS):
        A[c], B[c], C[c] = wa[:, i], wb[:, i], wc[:, i]
    m = lgb.LGBMClassifier(n_estimators=R["lgb"], learning_rate=0.02, max_depth=5, num_leaves=32,
                           min_child_samples=10, subsample=0.8, subsample_freq=1, colsample_bytree=0.3,
                           reg_alpha=0.071, reg_lambda=2.0, max_bin=1024, feature_pre_filter=False,
                           n_jobs=THREADS, verbose=-1, random_state=42)
    m.fit(A, y[a], eval_set=[(B, y[b])], eval_metric="auc", callbacks=[lgb.early_stopping(R["es"], verbose=False)])
    return {"D_megayak_liftwin": (m.predict_proba(B)[:, 1], m.predict_proba(C)[:, 1],
                                  f"trees {m.best_iteration_} cols {A.shape[1]}")}


register("Recipe D · megayak view C (lift + window)", prepare_megayak, fold_megayak, first_partition_min=15)

## 7 · Recipe E: bulsahkecici multi-resolution nested TE

The exact income and commute values get nested target rates at smoothing 10 / 1 / 100, plus log train
frequencies. Coarse keys (income ÷100, ÷1000, whole km) and every low-cardinality column get nested
rates at smoothing 1 / 10 / 100. The rates use `bincount` arithmetic, which is numerically the same as
the source's `groupby`. The inner split seed is `42 + 100 + fold` (fold counted from 1), as in the
source.

In [ ]:
E_TE_COLS = ["Annual_Income_USD", "Daily_Commute_km"]
E_DISCRETE = ["Age", "Number_of_Cars_Owned", "Charging_Stations_Near_Home",
              "Charging_Stations_Near_Work", "Environmental_Concern_Level"]


def prepare_bulsah():
    X, Xt = train_raw.drop(columns=[TARGET, ID]), test_raw.drop(columns=[ID])
    num = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
    cat = [c for c in X.columns if c not in num]
    both = pd.concat([X, Xt], ignore_index=True)
    dense = [both[num].fillna(X[num].median()).to_numpy(np.float32)]
    for c in cat:  # one-hot on train levels; unknown levels -> all zeros
        filled = both[c].fillna(X[c].mode().iloc[0])
        levels = sorted(X[c].dropna().unique())
        dense.append(np.stack([(filled == lv).to_numpy() for lv in levels], 1).astype(np.float32))
    M = np.hstack(dense)
    inc = np.rint(both.Annual_Income_USD.to_numpy(float)).astype(np.int64)
    km10 = np.rint(both.Daily_Commute_km.to_numpy(float) * 10).astype(np.int64)
    exact = {c: codes(both[c].to_numpy()) for c in E_TE_COLS}
    keys = {"income_floor_100": codes(inc // 100), "income_floor_1000": codes(inc // 1000),
            "commute_int": codes(km10 // 10)}
    for c in cat + E_DISCRETE:
        keys[f"cat_{c}"] = codes(both[c].astype(str).to_numpy())
    n = len(X)
    return dict(M=M[:n], Mt=M[n:], exact={c: (v[:n], v[n:]) for c, v in exact.items()},
                keys={c: (v[:n], v[n:]) for c, v in keys.items()})


def _rate(c_fit, y_fit, c_apply, ncode, m):
    sums = np.bincount(c_fit, weights=y_fit, minlength=ncode)
    cnt = np.bincount(c_fit, minlength=ncode)
    prior = float(y_fit.mean())
    return (sums[c_apply] + prior * m) / (cnt[c_apply] + m)


def nested_rate(c_train, c_test, tr, va, fold_seed, m):
    ncode = int(max(c_train.max(), c_test.max())) + 1
    ytr = y[tr].astype(float)
    ctr = c_train[tr]
    out_tr = np.zeros(len(tr))
    for i_tr, i_va in StratifiedKFold(5, shuffle=True, random_state=fold_seed).split(np.zeros(len(tr)), y[tr]):
        out_tr[i_va] = _rate(ctr[i_tr], ytr[i_tr], ctr[i_va], ncode, m)
    return out_tr, _rate(ctr, ytr, c_train[va], ncode, m), _rate(ctr, ytr, c_test, ncode, m)


def stack3(parts):
    """list of (train, valid, test) column triples -> three matrices"""
    return [np.column_stack(z) for z in zip(*parts)]


def fold_bulsah(ctx, k, tr, va):
    fold_seed = 42 + 100 + (k + 1)
    blocks = [stack3([nested_rate(*ctx["exact"][c], tr, va, fold_seed, 10.0) for c in E_TE_COLS])]
    freq = []
    for c in E_TE_COLS:
        c_train, c_test = ctx["exact"][c]
        cnt = np.bincount(c_train[tr], minlength=int(max(c_train.max(), c_test.max())) + 1)
        freq.append((np.log1p(cnt[c_train[tr]]), np.log1p(cnt[c_train[va]]), np.log1p(cnt[c_test])))
    blocks.append(stack3(freq))
    for m in (1.0, 100.0):
        blocks.append(stack3([nested_rate(*ctx["exact"][c], tr, va, fold_seed, m) for c in E_TE_COLS]))
    for m in (1.0, 10.0, 100.0):
        blocks.append(stack3([nested_rate(*ctx["keys"][c], tr, va, fold_seed, m) for c in ctx["keys"]]))
    A = np.hstack([ctx["M"][tr]] + [b[0] for b in blocks]).astype(np.float32)
    B = np.hstack([ctx["M"][va]] + [b[1] for b in blocks]).astype(np.float32)
    C = np.hstack([ctx["Mt"]] + [b[2] for b in blocks]).astype(np.float32)
    model = lgb.LGBMClassifier(objective="binary", n_estimators=R["lgb"], learning_rate=0.02, num_leaves=32,
                               max_depth=5, min_child_samples=10, subsample=0.80, subsample_freq=1,
                               colsample_bytree=0.30, reg_alpha=0.071, reg_lambda=2.0, max_bin=1024,
                               n_jobs=THREADS, deterministic=True, force_col_wise=True, verbosity=-1,
                               random_state=42 + (k + 1))
    model.fit(A, y[tr], eval_set=[(B, y[va])], eval_metric="auc",
              callbacks=[lgb.early_stopping(R["es"], verbose=False), lgb.log_evaluation(0)])
    return {"E_bulsah_mrte": (model.predict_proba(B)[:, 1], model.predict_proba(C)[:, 1],
                              f"trees {model.best_iteration_} cols {A.shape[1]}")}


register("Recipe E · bulsahkecici multi-res TE", prepare_bulsah, fold_bulsah, first_partition_min=20)

## 8 · Run: partition → recipe → fold

In [ ]:
for p_idx, seed in enumerate(PARTITION_SEEDS):
    print(f"\n{'#' * 90}\nPARTITION {p_idx + 1}/{len(PARTITION_SEEDS)} · fold seed {seed}   ({elapsed()})\n{'#' * 90}")
    for group in GROUPS:
        if group["active"]:
            run_partition(group, seed)
for group in GROUPS:
    group["ctx"] = None
gc.collect()
for name, r in RESULTS.items():
    r["oof"] = np.mean(r["oofs"], axis=0)
    r["test"] = np.mean(r["tests"], axis=0)
    r["auc"] = float(roc_auc_score(y, r["oof"]))

## 9 · Component scoreboard

In [ ]:
if not RESULTS:
    raise RuntimeError("No component finished inside the budget")
names = sorted(RESULTS, key=lambda n: -RESULTS[n]["auc"])
board = pd.DataFrame([{"component": n, "oof_auc": RESULTS[n]["auc"],
                       "fold_min": min(RESULTS[n]["fold_auc"]), "fold_max": max(RESULTS[n]["fold_auc"]),
                       "partitions": len(RESULTS[n]["oofs"]),
                       "single_partition_mean": float(np.mean(RESULTS[n]["partition_auc"])),
                       "minutes": RESULTS[n]["seconds"] / 60} for n in names])
print(board.to_string(index=False, float_format=lambda v: f"{v:.6f}"))
P = np.column_stack([RESULTS[n]["oof"] for n in names])
PT = np.column_stack([RESULTS[n]["test"] for n in names])
sub_rows = np.random.default_rng(0).choice(len(y), min(200_000, len(y)), replace=False)
rank_corr = pd.DataFrame(np.corrcoef(np.apply_along_axis(rankdata, 0, P[sub_rows]), rowvar=False),
                         index=names, columns=names)
print("\nOOF Spearman correlation (200k-row sample):")
print(rank_corr.round(4).to_string())

## 10 · Nested-gated logit blend

Caruana ensemble selection with replacement, on logits. For each outer fold the weights are chosen on
the other four folds and scored on the held-out fold, against the best single component chosen the
same way. The pooled weights ship only if the blend wins at least 4 of 5 folds with a positive mean
gain. Otherwise the best single component ships.

In [ ]:
Lg = logit(np.clip(P, 1e-7, 1 - 1e-7))
LgT = logit(np.clip(PT, 1e-7, 1 - 1e-7))


def fast_auc(yv, s):
    r = rankdata(s)
    npos = yv.sum()
    return (r[yv == 1].sum() - npos * (npos + 1) / 2) / (npos * (len(yv) - npos))


def hill_climb(L, yv, steps=60):
    k = L.shape[1]
    single = np.array([fast_auc(yv, L[:, j]) for j in range(k)])
    counts = np.zeros(k)
    counts[single.argmax()] = 1
    cur, cur_auc = L[:, single.argmax()].copy(), single.max()
    for _ in range(steps):
        n = counts.sum()
        trial = [fast_auc(yv, (cur * n + L[:, j]) / (n + 1)) for j in range(k)]
        j = int(np.argmax(trial))
        if trial[j] <= cur_auc + 1e-7:
            break
        counts[j] += 1
        cur, cur_auc = (cur * n + L[:, j]) / (n + 1), trial[j]
    return counts / counts.sum(), single


gate = []
for k, (tr, va) in enumerate(GATE_FOLDS):
    w, single = hill_climb(Lg[tr], y[tr])
    j = int(single.argmax())
    b_auc = fast_auc(y[va], Lg[va] @ w)
    s_auc = fast_auc(y[va], Lg[va, j])
    gate.append(dict(fold=k, blend_auc=b_auc, best_single=names[j], single_auc=s_auc, gain=b_auc - s_auc,
                     weights={names[i]: round(float(v), 3) for i, v in enumerate(w) if v > 0}))
gate_df = pd.DataFrame(gate)
print(gate_df.drop(columns="weights").to_string(index=False, float_format=lambda v: f"{v:.6f}"))
wins = int((gate_df.gain > 0).sum())
use_blend = wins >= 4 and gate_df.gain.mean() > 0
W, single_all = hill_climb(Lg, y)
if use_blend:
    final_oof, final_test = Lg @ W, LgT @ W
    chosen = {names[i]: float(v) for i, v in enumerate(W) if v > 0}
else:
    j = int(single_all.argmax())
    final_oof, final_test = Lg[:, j], LgT[:, j]
    chosen = {names[j]: 1.0}
print(f"\nnested gate: blend wins {wins}/5, mean gain {gate_df.gain.mean():+.6f} -> "
      f"{'BLEND' if use_blend else 'BEST SINGLE'}")
print("final weights:", {k_: round(v, 3) for k_, v in chosen.items()})
print(f"final OOF AUC {roc_auc_score(y, final_oof):.6f} | equal-weight logit mean "
      f"{roc_auc_score(y, Lg.mean(1)):.6f} | best single {single_all.max():.6f}")

## 11 · Export

In [ ]:
pred = expit(final_test)
assert len(pred) == len(test_ids) and np.isfinite(pred).all()
submission = sample[[ID]].merge(pd.DataFrame({ID: test_ids, TARGET: pred}), on=ID, how="left")
assert submission[TARGET].notna().all() and len(submission) == len(sample)
submission.to_csv("submission.csv", index=False)

oof_table = pd.DataFrame({ID: train_raw[ID], "fold": fold_id, TARGET: y,
                          **{n: RESULTS[n]["oof"].astype("float32") for n in names}, "fusion": expit(final_oof)})
test_table = pd.DataFrame({ID: test_ids, **{n: RESULTS[n]["test"].astype("float32") for n in names}, "fusion": pred})
oof_table.to_csv(OUT / "oof_components.csv", index=False, float_format="%.7f")
test_table.to_csv(OUT / "test_components.csv", index=False, float_format="%.7f")
report = dict(components=board.to_dict("records"), gate=gate, blend_used=bool(use_blend), weights=chosen,
              final_oof_auc=float(roc_auc_score(y, final_oof)), minutes=(time.time() - T0) / 60,
              gpu=GPU, smoke=SMOKE, folds=NFOLD, partition_seeds=list(PARTITION_SEEDS))
(OUT / "fusion_report.json").write_text(json.dumps(report, indent=2, default=float))
print(submission.head())
print(f"\nwrote submission.csv ({len(submission):,} rows) | total {elapsed()}")

## Takeaways

- Every component is trained here. Nothing is copied from public predictions.
- Partition bagging averages 20 fold-models per recipe. This mainly stabilises the test predictions.
  The bag OOF AUC above shows how much it also gains out-of-fold.
- The blend still has to beat the best single recipe on held-out folds before it ships.
- `fusion_outputs/` holds the bagged OOF and test vector of every component.